# Изазов: Анализа текста о Науци о подацима

У овом примеру, хајде да урадимо једноставну вежбу која обухвата све кораке традиционалног процеса науке о подацима. Не морате писати никакав код, можете само кликнути на ћелије испод да их извршите и посматрате резултат. Као изазов, препоручује се да испробате овај код са различитим подацима.

## Циљ

У овој лекцији смо дискутовали о различитим појмовима у вези са Науком о подацима. Хајде да покушамо да откријемо више повезаних појмова тако што ћемо урадити неку **текстуалну рударску анализу**. Почећемо са текстом о Науци о подацима, извући кључне речи из њега, а затим ћемо покушати да визуализујемо резултат.

Као текст ћу користити страницу о Науци о подацима са Википедије:


In [ ]:
url = 'https://en.wikipedia.org/wiki/Data_science'

## Корак 1: Добијање података

Први корак у сваком процесу науке о подацима је добијање података. За то ћемо користити библиотеку `requests`:


In [ ]:
import requests

# Define a custom header.
headers = {
    'User-Agent': 'DataScienceChallenge/1.0 (myemail@gmail.com)'
}

# Pass the headers into the get request
response = requests.get(url, headers=headers)

if response.status_code == 200:
    text = response.content.decode('utf-8')
    print(text[:1000])
else:
    print(f"Error: {response.status_code}")

## Корак 2: Претварање података

Следећи корак је претворити податке у облик погодан за обраду. У нашем случају, преузели смо HTML изворни код са странице и нужно је да га претворимо у обичан текст.

Постоји много начина на које се то може урадити. Ми ћемо користити [BeautifulSoup](https://www.crummy.com/software/BeautifulSoup/), популарну Python библиотеку за парсирање HTML-а. BeautifulSoup нам омогућава да циљамо одређене HTML елементе, тако да можемо да се фокусирамо на главни садржај чланка из Википедије и смањимо неке навигационе меније, бочне траке, подножја и други небитни садржај (иако ће неки шаблонски текст можда и даље остати).


Прво, потребно је да инсталирамо BeautifulSoup библиотеку за парсирање HTML-а:


In [ ]:
import sys
!{sys.executable} -m pip install beautifulsoup4

In [ ]:
from bs4 import BeautifulSoup

# Parse the HTML content
soup = BeautifulSoup(text, 'html.parser')

# Extract only the main article content from Wikipedia
# Wikipedia uses 'mw-parser-output' class for the main article content
content = soup.find('div', class_='mw-parser-output')

def clean_wikipedia_content(content_node):
    """Remove common non-article elements from a Wikipedia content node."""
    # Strip jump links, navboxes, reference lists/superscripts, edit sections, TOC, sidebars, etc.
    selectors = [
        '.mw-jump-link',
        '.navbox',
        '.reflist',
        'sup.reference',
        '.mw-editsection',
        '.hatnote',
        '.metadata',
        '.infobox',
        '#toc',
        '.toc',
        '.sidebar',
    ]
    for selector in selectors:
        for el in content_node.select(selector):
            el.decompose()

if content:
    # Clean the content node to better approximate article text only.
    clean_wikipedia_content(content)
    text = content.get_text(separator=' ', strip=True)
    print(text[:1000])
else:
    print("Could not find main content. Using full page text.")
    text = soup.get_text(separator=' ', strip=True)
    print(text[:1000])

## Корак 3: Добијање увида

Најважнији корак је претварање наших података у неки облик из ког можемо извући увиде. У нашем случају, желимо да издвојимо кључне речи из текста и видимо које кључне речи су значајније.

Користићемо Python библиотеку под називом [RAKE](https://github.com/aneesha/RAKE) за издвајање кључних речи. Прво, хајде да инсталирамо ову библиотеку уколико није присутна: 


In [ ]:
import sys
!{sys.executable} -m pip install nlp_rake

Главна функционалност је доступна преко објекта `Rake`, који можемо прилагодити коришћењем неких параметара. У нашем случају, подесићемо минималну дужину кључне речи на 5 знакова, минималну учесталост кључне речи у документу на 3, и максималан број речи у кључној речи на 2. Слободно експериментишите са другим вредностима и посматрајте резултат.


In [ ]:
import nlp_rake
extractor = nlp_rake.Rake(max_words=2,min_freq=3,min_chars=5)
res = extractor.apply(text)
res


Добијена је листа термина заједно са повезаним степеном значаја. Као што видите, најрелевантније дисциплине, као што су машинско учење и велики подаци, присутне су на врху листе.

## Четврти корак: Визуализација резултата

Људи најбоље тумаче податке у визуелном облику. Због тога често има смисла визуализовати податке како бисмо извукли неке увиде. Можемо користити библиотеку `matplotlib` у Питону за приказ једноставне дистрибуције кључних речи са њиховом релевантношћу:


In [ ]:
import matplotlib.pyplot as plt

def plot(pair_list):
    k,v = zip(*pair_list)
    plt.bar(range(len(k)),v)
    plt.xticks(range(len(k)),k,rotation='vertical')
    plt.show()

plot(res)

Међутим, постоји и бољи начин за визуелизацију учесталости речи - коришћењем **Облака речи**. Биће нам потребно да инсталирамо другу библиотеку како бисмо исцртали облак речи из наше листе кључних речи.


In [ ]:
!{sys.executable} -m pip install wordcloud

Објекат `WordCloud` је одговоран за прихватање оригиналног текста или унапред израчунате листе речи са њиховим фреквенцијама, и враћа слику која се затим може приказати коришћењем `matplotlib`:


In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

wc = WordCloud(background_color='white',width=800,height=600)
plt.figure(figsize=(15,7))
plt.imshow(wc.generate_from_frequencies({ k:v for k,v in res }))

Такође можемо проследити оригинални текст у `WordCloud` - хајде да видимо да ли можемо добити сличан резултат:


In [ ]:
plt.figure(figsize=(15,7))
plt.imshow(wc.generate(text))

In [ ]:
wc.generate(text).to_file('images/ds_wordcloud.png')

Видећете да облак речи сада изгледа импресивније, али садржи и пуно буке (нпр. неспојиве речи као што је `Retrieved on`). Такође, добијамо мање кључних речи које се састоје из две речи, као што су *data scientist* или *computer science*. То је зато што алгоритам RAKE много боље бира добре кључне речи из текста. Овај пример илуструје важност претходне обраде и чишћења података, јер нам јасна слика на крају омогућава доношење бољих одлука.

У овом задатку прошли смо кроз једноставан процес извлачења смисла из текста Википедије, у облику кључних речи и облака речи. Овај пример је прилично једноставан, али добро демонстрира све типичне кораке које један дата сајентиста предузима када ради са подацима, почев од прибављања података па до визуализације.

На нашем курсу ћемо детаљно размотрити све те кораке.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Изјава о одрицању одговорности**:
Овај документ је преведен коришћењем услуге за аутоматски превод [Co-op Translator](https://github.com/Azure/co-op-translator). Иако тежимо тачности, имајте у виду да аутоматски преводи могу садржати грешке или нетачности. Оригинални документ на његовом изворном језику треба сматрати ауторитативним извором. За критичне информације препоручује се професионални људски превод. Нисмо одговорни за било каква неспоразума или погрешна тумачења која произилазе из коришћења овог превода.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
